**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**04 | Classical, Dual-Branch and Attention-Based Fusion**

![Classical, Dual-Branch and Attention-Based Fusion workflow](../../images/04_fusion_and_modality_ablations.svg)

**Research objective:** Test whether paired acceleration adds information to sEMG under exactly matched participant-disjoint Development evaluation.

**Dataset:** The original aligned E1 window registry and unimodal out-of-fold class probabilities from Notebooks 02 and 03.

**Notebook scope:** Compare feature-level, fixed-weight decision-level and trained neural fusion, followed by clearly labelled synthetic sensor perturbations.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path
import time

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------ 5. Deep Learning and Compute Device ---------------------
import torch
from core.computer_device import computer_device
device = computer_device(min_vram_gb=None)

# ------------------------ 6. Study Research Utilities ------------------------
from ninapro_dataset import ensure_numerical_cache
from ninapro_evaluation import (
    build_classical, classical_features, aligned_probabilities,
    participant_metrics, select_model, predictions_from_probabilities,
)
from ninapro_models import FUSION_MODELS, UNIMODAL_MODELS
from ninapro_training import fit_neural, predict_neural

# ---------------------- 7. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Recover Common Cohort, Folds and Unimodal Evidence**

Read the same window row ordering and participant-fold assignments used for every unimodal comparison. Recover saved Development out-of-fold probabilities with their original class order. A downstream fusion comparison must not join modalities by row order without the canonical source key.

In [ ]:
# ========== 02. Recover Common Cohort, Folds and Unimodal Evidence ===========

NOTEBOOK_NUMBER = 4
SEED = 42
index = pd.read_csv(output_dir / "02_window_index.csv")
roles = pd.read_csv(output_dir / "02_participant_split_assignments.csv")
folds = pd.read_csv(output_dir / "02_development_fold_assignments.csv")
classes = np.sort(pd.read_csv(output_dir / "01_label_dictionary.csv").label.to_numpy(int))
index["role"] = index.subject_id.map(dict(zip(roles.subject_id, roles.role)))
index["validation_fold"] = index.subject_id.map(dict(zip(folds.subject_id, folds.validation_fold)))
dev = index.loc[index.role == "Development"].copy()
common_folds = sorted(dev.validation_fold.dropna().unique().astype(int))
unimodal_metrics = pd.read_csv(output_dir / "03_all_unimodal_metrics.csv")
with np.load(output_dir / "03_oof_probabilities.npz") as original:
    previous = {key: original[key] for key in original.files}
for name, proba in previous.items():
    if proba.shape != (len(index), len(classes)):
        raise ValueError(f"Out-of-fold probability row/class mismatch: {name}")
print("Identical Development people:", dev.subject_id.nunique())
print("Paired windows:", len(dev))


**03. Prepare Source-Aligned Feature and Neural Inputs**

Classical early fusion concatenates sEMG and acceleration features from precisely the same `window_row`. Neural fusion reads two separate source-channel matrices for that window. These representations are generated only from the same original exercise time interval and source person.

In [ ]:
# =========== 03. Prepare Source-Aligned Feature and Neural Inputs ============

emg_features, acc_features = ensure_numerical_cache(root, index, allow_source_download=True)
assert index.window_key.is_unique
assert len(emg_features) == len(acc_features) == len(index)
print("Early-fusion feature width:", emg_features.shape[1] + acc_features.shape[1])


**04. Evaluate Classical Early and Fixed-Weight Late Fusion**

For early fusion, fit Logistic Regression or Random Forest on concatenated features using Training people only. For late fusion, average the matching, already out-of-fold unimodal probability vectors using a predeclared 0.5/0.5 weight and identical source labels. No outer Validation outcomes select mixture weights or fit calibration.

In [ ]:
# ========= 04. Evaluate Classical Early and Fixed-Weight Late Fusion =========

CLASSICAL_FUSION = ("F1_logreg", "F1_rf", "F2_logreg", "F2_rf")
fusion_probs = {key: np.full((len(index), len(classes)), np.nan, np.float32)
                for key in (*CLASSICAL_FUSION, "F4_deep_late", *FUSION_MODELS)}
classical_results, neural_results, histories, ablation_results = [], [], [], []
for fold in common_folds:
    fitting = dev.loc[dev.validation_fold != fold]
    validation = dev.loc[dev.validation_fold == fold]
    for name in ("F1_logreg", "F1_rf"):
        started = time.perf_counter()
        model = build_classical(name, seed=SEED)
        model.fit(classical_features(name, emg_features[fitting.window_row],
                                     acc_features[fitting.window_row]),
                  fitting.label.to_numpy(int))
        proba = aligned_probabilities(model, classical_features(
            name, emg_features[validation.window_row],
            acc_features[validation.window_row]), classes)
        fusion_probs[name][validation.window_row.to_numpy(int)] = proba
        metric = participant_metrics(validation.reset_index(drop=True), proba,
                                     classes, model_id=name, fold=fold, seed=SEED)
        metric["fit_and_prediction_seconds"] = time.perf_counter() - started
        classical_results.append(metric)
    print(f"Classical early-fusion fold {fold} complete")
for model_family in ("logreg", "rf"):
    name = f"F2_{model_family}"
    fusion_probs[name] = 0.5 * (previous[f"B1_{model_family}"] +
                               previous[f"B2_{model_family}"])
    for fold in common_folds:
        validation = dev.loc[dev.validation_fold == fold]
        proba = fusion_probs[name][validation.window_row.to_numpy(int)]
        classical_results.append(participant_metrics(
            validation.reset_index(drop=True), proba, classes,
            model_id=name, fold=fold, seed=SEED))
classical_metrics = pd.concat(classical_results, ignore_index=True)


**05. Construct Deep Decision-Level Fusion**

Select one sEMG neural branch and one accelerometer neural branch using their person-macro Development evidence. Their matched out-of-fold class probabilities are combined with equal weight. This is a true decision-fusion comparison; its components were independently trained in Notebook 03.

In [ ]:
# ================= 05. Construct Deep Decision-Level Fusion ==================

chosen_emg, _ = select_model(unimodal_metrics,
                            [x for x in UNIMODAL_MODELS if "_emg_" in x])
chosen_acc, _ = select_model(unimodal_metrics,
                            [x for x in UNIMODAL_MODELS if "_acc_" in x])
fusion_probs["F4_deep_late"] = 0.5 * (previous[chosen_emg] + previous[chosen_acc])
for fold in common_folds:
    validation = dev.loc[dev.validation_fold == fold]
    proba = fusion_probs["F4_deep_late"][validation.window_row.to_numpy(int)]
    classical_results.append(participant_metrics(
        validation.reset_index(drop=True), proba, classes,
        model_id="F4_deep_late", fold=fold, seed=SEED))
print("Fixed-weight neural late-fusion components:", chosen_emg, "+", chosen_acc)


**06. Train Dual-Branch CNN, TCN and Cross-Modal Fusion**

Dual encoders learn modality-specific temporal representations on exactly paired windows. The concatenation baseline, gated weighting and bounded two-head cross-attention are independently trained in each grouped fold. Internal early stopping remains strictly within outer-fold Training people. Parameter counts and actual fitting times make architectural capacity explicit.

In [ ]:
# =========== 06. Train Dual-Branch CNN, TCN and Cross-Modal Fusion ===========

NEURAL_EPOCHS = 10
NEURAL_PATIENCE = 3
for fold in common_folds:
    fitting = dev.loc[dev.validation_fold != fold].copy()
    validation = dev.loc[dev.validation_fold == fold].copy()
    for name in FUSION_MODELS:
        model, stats, history, info = fit_neural(
            name, fitting, processed_dir, classes, device=device,
            seed=SEED, epochs=NEURAL_EPOCHS,
            patience=NEURAL_PATIENCE,
        )
        proba = predict_neural(name, model, validation, processed_dir,
                               classes, stats, device=device)
        fusion_probs[name][validation.window_row.to_numpy(int)] = proba
        metric = participant_metrics(validation.reset_index(drop=True), proba,
                                     classes, model_id=name, fold=fold,
                                     seed=SEED)
        for key, value in info.items():
            if key not in ("seed", "model_id"):
                metric[key] = value
        neural_results.append(metric)
        history["validation_fold"] = fold
        histories.append(history)
        if name in ("F3_dual_cnn", "F5_cross_attention"):
            for condition in ("acc_missing_mean", "acc_sensor_01_missing_mean",
                              "acc_contiguous_gap_mean"):
                altered = predict_neural(
                    name, model, validation, processed_dir, classes, stats,
                    device=device, perturbation=condition,
                )
                measured = participant_metrics(validation.reset_index(drop=True),
                    altered, classes, model_id=name, fold=fold, seed=SEED)
                measured["perturbation"] = condition
                ablation_results.append(measured)
        del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"Deep-fusion grouped fold {fold} complete")
neural_metrics = pd.concat(neural_results, ignore_index=True)
pd.concat(histories, ignore_index=True).to_csv(
    output_dir / "04_neural_learning_history.csv", index=False)
robustness = pd.concat(ablation_results, ignore_index=True)
robustness.to_csv(output_dir / "04_fusion_robustness.csv", index=False)


**07. Compare Paired Development Effects and Freeze Recipes**

A single Development-selected unimodal reference is fixed before Test. All fusion models are compared against this same reference on the same participant windows. The preferred fusion is selected by mean participant-macro F1 across the common folds, with fixed class order and no Test-based choice. The selection file records any late-fusion component identities needed for final refitting.

In [ ]:
# ========= 07. Compare Paired Development Effects and Freeze Recipes =========

classical_metrics = pd.concat(classical_results, ignore_index=True)
all_fusion_metrics = pd.concat([classical_metrics, neural_metrics], ignore_index=True)
classical_metrics.loc[classical_metrics.model_id.str.startswith("F1_") |
                      classical_metrics.model_id.str.startswith("F2_")].to_csv(
    output_dir / "04_dev_fusion_metrics.csv", index=False)
all_fusion_metrics.loc[all_fusion_metrics.model_id.isin(["F4_deep_late", *FUSION_MODELS])].to_csv(
    output_dir / "04_dev_deep_fusion_metrics.csv", index=False)
all_fusion_metrics.to_csv(output_dir / "04_all_fusion_metrics.csv", index=False)
best_unimodal, u_summary = select_model(unimodal_metrics, sorted(previous))
best_fusion, f_summary = select_model(all_fusion_metrics, sorted(fusion_probs))
selection = pd.DataFrame([
    {"selection_role": "unimodal_comparator", "model_id": best_unimodal,
     "component_emg": "", "component_acc": ""},
    {"selection_role": "fusion_comparator", "model_id": best_fusion,
     "component_emg": chosen_emg if best_fusion == "F4_deep_late" else "",
     "component_acc": chosen_acc if best_fusion == "F4_deep_late" else ""},
])
selection.to_csv(output_dir / "04_model_selection.csv", index=False)
per_person = []
for name, proba in fusion_probs.items():
    matched = dev.reset_index(drop=True)
    p = participant_metrics(matched, proba[dev.window_row.to_numpy(int)],
                            classes, model_id=name)
    u = participant_metrics(matched,
        previous[best_unimodal][dev.window_row.to_numpy(int)], classes,
        model_id=best_unimodal).set_index("subject_id")
    p["unimodal_reference"] = best_unimodal
    p["unimodal_macro_f1"] = p.subject_id.map(u.macro_f1)
    p["fusion_minus_unimodal_macro_f1"] = p.macro_f1 - p.unimodal_macro_f1
    per_person.append(p)
paired_effects = pd.concat(per_person, ignore_index=True)
paired_effects.to_csv(output_dir / "04_oof_paired_differences.csv", index=False)
np.savez_compressed(output_dir / "04_oof_probabilities.npz", **fusion_probs)
print("Development-selected reference:", best_unimodal)
print("Development-selected fusion:", best_fusion)
display(selection)
display(f_summary)


**08. Assess Neural Fusion Stability Across Three Seeds**

The strongest Development deep-fusion architecture is repeated using two additional fixed seeds on the same independent Development folds. These runs quantify training-seed variation separately from participant variation, and do not modify the primary Development-selected model or final Test configuration.

In [ ]:
# =========== 08. Assess Neural Fusion Stability Across Three Seeds ===========

RUN_FINALIST_ROBUSTNESS = True
EXTRA_SEEDS = (43, 44)
best_deep, _ = select_model(neural_metrics, list(FUSION_MODELS))
if RUN_FINALIST_ROBUSTNESS:
    extra_metrics, extra_curves = [], []
    for replicate_seed in EXTRA_SEEDS:
        for fold in common_folds:
            fitting = dev.loc[dev.validation_fold != fold]
            validation = dev.loc[dev.validation_fold == fold]
            model, stats, history, info = fit_neural(
                best_deep, fitting, processed_dir, classes, device=device,
                seed=replicate_seed, epochs=NEURAL_EPOCHS,
                patience=NEURAL_PATIENCE,
            )
            probs = predict_neural(best_deep, model, validation,
                                   processed_dir, classes, stats, device=device)
            scores = participant_metrics(validation.reset_index(drop=True),
                probs, classes, model_id=best_deep, fold=fold, seed=replicate_seed)
            scores["fit_seconds"] = info["fit_seconds"]
            extra_metrics.append(scores)
            history["validation_fold"] = fold
            extra_curves.append(history)
            del model
    pd.concat(extra_metrics, ignore_index=True).to_csv(
        output_dir / "04_deep_multiseed_metrics.csv", index=False)
    pd.concat(extra_curves, ignore_index=True).to_csv(
        output_dir / "04_deep_multiseed_history.csv", index=False)
    print("Completed additional seeds for deep-fusion finalist:", best_deep)


**09. Plot Fusion Benefits and Simulated Sensor Failure**

The fusion comparison uses equal participant weighting. The perturbation panel reports the response of pre-trained fusion models to synthetic mean-replacement of acceleration channels; it must not be mistaken for validation of a physically missing sensor or a retrained modality-absent model.

In [ ]:
# =========== 09. Plot Fusion Benefits and Simulated Sensor Failure ===========

fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
order = f_summary.model_id.tolist()
ax.barh(order[::-1], f_summary.mean_subject_macro_f1.to_numpy()[::-1])
ax.axvline(float(u_summary.set_index("model_id").loc[best_unimodal, "mean_subject_macro_f1"]),
           color="0.35", ls="--", label=f"Best unimodal: {best_unimodal}")
ax.set(xlabel="Development participant-mean macro F1", title="Matched multimodal fusion comparison")
ax.grid(axis="x", alpha=.2)
ax.legend()
save_figure(fig, output_dir / "04_fusion_comparison.png")
plt.show()
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
for name, group in robustness.groupby("model_id"):
    means = group.groupby("perturbation").macro_f1.mean()
    ax.plot(means.index, means.to_numpy(), marker="o", label=name)
ax.set(xlabel="Simulated acceleration perturbation", ylabel="Development participant-mean macro F1",
       title="Response to synthetic channel replacement")
ax.tick_params(axis="x", rotation=16)
ax.grid(alpha=.2)
ax.legend()
save_figure(fig, output_dir / "04_modality_sensitivity.png")
plt.show()


**10. Current Outputs and Scientific Interpretation**

The Development-selected unimodal and fusion recipes, paired individual differences and training configurations define the fixed models carried into Notebook 05. Any final Test interpretation must consider model capacity, observed uncertainty and synthetic-perturbation limitations.

In [ ]:
# ============= 10. Current Outputs and Scientific Interpretation =============

display(pd.DataFrame({"output_file": [
    p.name for p in sorted(output_dir.glob("%02d_*" % NOTEBOOK_NUMBER))
    if p.is_file()
]}))
download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
